In [ ]:
if 'spark' in globals():
    spark.stop()

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("postgres_to_parquet_silver") \
    .master("spark://spark-master:7077") \
    .config("spark.default.parallelism", "4") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.autoBroadcastJoinThreshold", "10m") \
    .config("spark.sql.parquet.enableVectorizedReader", "true") \
    .config("spark.sql.legacy.parquet.nanosAsLong", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/09 16:28:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:

STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "mongo"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-09-06-Sep"
TABLE_NAME = "videos"

MONGO_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}/{TABLE_NAME}.parquet"

In [4]:
mongo_df = spark.read.parquet(MONGO_BRONZE_PATH)

26/09/09 16:29:05 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [5]:
STORAGE_PROTOCOL = "s3a://"
BUCKET_NAME = "end-to-end-streaming-data-platform-bronze"
SOURCE_SUSTEM = "kafka"
FOLDER_NAME = "ingestion_data"
execution_date = "2026-09-06-Sep"



KAFKA_BRONZE_PATH = f"{STORAGE_PROTOCOL}{BUCKET_NAME}/{SOURCE_SUSTEM}/{FOLDER_NAME}={execution_date}"

#Pv = "s3a://end-to-end-streaming-data-platform-bronze/mongo/ingestion_data=2026-07-13-Jul/videos.parquet"

In [6]:


print("=" * 70)
print("KAFKA BRONZE — EDA")
print("=" * 70)



kafka_df = spark.read.parquet(KAFKA_BRONZE_PATH)

print("Kafka Bronze loaded successfully.")

KAFKA BRONZE — EDA
Kafka Bronze loaded successfully.


In [ ]:
pk_tests = {
    "dim_videos": "video_id",
    "fact_events": "event_id",
}

In [7]:
relationships = [
    (
        kafka_df,
        "video_id",
        mongo_df,
        "video_id",
    ),
]

for child_df, child_fk, parent_df, parent_pk in relationships:

    orphan_count = (
        child_df
        .join(
            parent_df.select(parent_pk),
            child_df[child_fk] == parent_df[parent_pk],
            "left_anti",
        )
        .count()
    )

    print(
        f"{child_fk} -> {parent_pk} | "
        f"orphans: {orphan_count}"
    )

video_id -> video_id | orphans: 0


In [7]:
kafka_df \
    .join(
        mongo_df.select("video_id"),
        kafka_df["video_id"] == mongo_df["video_id"],
        "left_anti"
    ) \
    .count()

0

In [7]:


videos_df = mongo_df.select(
    "video_id",
    "duration_sec",
)


check_df = (
    kafka_df
    .join(videos_df, on="video_id", how="left")
    .select(
        "video_id",
        "interaction_type",
        "watch_time_sec",
        "duration_sec",
    )
)


invalid_df = check_df.filter(
    (
        (F.col("duration_sec") > 0) &
        (F.col("watch_time_sec") > F.col("duration_sec"))
    )
    |
    (
        F.col("duration_sec").isNull() &
        F.col("watch_time_sec").isNotNull()
    )
)

print("Total interactions:", check_df.count())
print("Invalid watch_time:", invalid_df.count())

invalid_df.show(20, truncate=False)

Total interactions: 15680
Invalid watch_time: 0
+--------+----------------+--------------+------------+
|video_id|interaction_type|watch_time_sec|duration_sec|
+--------+----------------+--------------+------------+
+--------+----------------+--------------+------------+



In [2]:
# ============================================================
# 1. LOAD SILVER DATASETS ONCE
# ============================================================

df_users_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/users.parquet")
df_subscriptions_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/subscription.parquet")
#df_payments_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/postgres/payments.parquet")
df_videos_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/mongo/videos_silver.parquet")
df_events_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/kafka/events_silver.parquet")


26/09/03 18:30:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [7]:
df_videos_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/mongo/videos_silver.parquet")
#df_kafka_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/kafka/events_silver.parquet")


In [2]:
df_kafka_silver = spark.read.parquet("s3a://end-to-end-streaming-data-platform-silver/kafka/events_silver.parquet")

26/09/03 13:57:06 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

In [3]:
events_with_valid_time = df_kafka_silver.filter(
    F.col("event_timestamp").isNotNull()
)

subscriptions = df_subscriptions_silver.select(
    "user_id",
    "subscription_id",
    "start_date",
    "end_date"
)

events_outside_subscription = (
    events_with_valid_time.alias("e")
    .join(
        subscriptions.alias("s"),
        (
            (F.col("e.user_id") == F.col("s.user_id"))
            & (F.col("e.event_timestamp") >= F.col("s.start_date"))
            & (
                F.col("s.end_date").isNull()
                | (F.col("e.event_timestamp") <= F.col("s.end_date"))
            )
        ),
        "left"
    )
    .groupBy("e.event_id")
    .agg(
        F.count("s.subscription_id").alias("matching_subscriptions")
    )
    .filter(F.col("matching_subscriptions") == 0)
)

events_outside_subscription.agg(
    F.count("*").alias("events_outside_subscription_period")
).show()

+----------------------------------+
|events_outside_subscription_period|
+----------------------------------+
|                                 0|
+----------------------------------+



In [4]:
final_temporal_audit = {
    "users": df_users_silver.select(
        F.min("created_at").alias("min"),
        F.max("created_at").alias("max"),
        F.countDistinct("created_at").alias("distinct")
    ).collect()[0].asDict(),

    "subscriptions_before_user": (
        df_subscriptions_silver.alias("s")
        .join(
            df_users_silver.select("user_id", "created_at").alias("u"),
            "user_id",
            "left"
        )
        .filter(F.col("s.start_date") < F.col("u.created_at"))
        .count()
    ),

    "payments_before_subscription": (
        df_payments_silver.alias("p")
        .join(
            df_subscriptions_silver.select(
                "subscription_id", "start_date", "end_date"
            ).alias("s"),
            "subscription_id",
            "left"
        )
        .filter(F.col("p.payment_date") < F.col("s.start_date"))
        .count()
    ),

    "payments_after_subscription": (
        df_payments_silver.alias("p")
        .join(
            df_subscriptions_silver.select(
                "subscription_id", "start_date", "end_date"
            ).alias("s"),
            "subscription_id",
            "left"
        )
        .filter(
            F.col("s.end_date").isNotNull()
            & (F.col("p.payment_date") > F.col("s.end_date"))
        )
        .count()
    ),

    "events_before_user": (
        df_kafka_silver.alias("k")
        .join(df_users_silver.select("user_id", "created_at"), "user_id")
        .filter(F.col("k.event_timestamp") < F.col("created_at"))
        .count()
    ),

    "events_before_video": (
        df_kafka_silver.alias("k")
        .join(df_videos_silver.select("video_id", "created_at"), "video_id")
        .filter(F.col("k.event_timestamp") < F.col("created_at"))
        .count()
    ),

    "orphan_users": (
        df_kafka_silver
        .join(df_users_silver.select("user_id"), "user_id", "left_anti")
        .count()
    ),

    "orphan_videos": (
        df_kafka_silver
        .join(df_videos_silver.select("video_id"), "video_id", "left_anti")
        .count()
    ),
}

final_temporal_audit

{'users': {'min': datetime.datetime(2026, 1, 1, 0, 0),
  'max': datetime.datetime(2031, 5, 24, 0, 23, 13),
  'distinct': 10000},
 'subscriptions_before_user': 0,
 'payments_before_subscription': 0,
 'payments_after_subscription': 0,
 'events_before_user': 0,
 'events_before_video': 0,
 'orphan_users': 0,
 'orphan_videos': 0}

In [5]:
dirty_preservation = {
    "users_login_before_creation": df_users_silver.filter(
        F.col("is_login_before_creation")
    ).count(),

    "subscriptions_end_before_start": df_subscriptions_silver.filter(
        F.col("is_end_date_before_start_date")
    ).count(),

    "videos_missing_title": df_videos_silver.filter(
        F.col("is_missing_title")
    ).count(),

    "videos_invalid_duration": df_videos_silver.filter(
        F.col("is_invalid_duration")
    ).count(),

    "videos_live_without_match": df_videos_silver.filter(
        F.col("is_live_without_match_id")
    ).count(),

    "videos_likes_gt_views": df_videos_silver.filter(
        F.col("is_likes_greater_than_views")
    ).count(),

    "kafka_invalid_timestamp": df_kafka_silver.filter(
        F.col("dq_invalid_timestamp")
    ).count(),

    "kafka_invalid_watch_time": df_kafka_silver.filter(
        F.col("dq_invalid_watch_time")
    ).count(),

    "kafka_invalid_interaction": df_kafka_silver.filter(
        F.col("dq_invalid_interaction")
    ).count(),

    "kafka_invalid_device": df_kafka_silver.filter(
        F.col("dq_invalid_device")
    ).count(),
}

dirty_preservation

{'users_login_before_creation': 1,
 'subscriptions_end_before_start': 1,
 'videos_missing_title': 1,
 'videos_invalid_duration': 2,
 'videos_live_without_match': 1,
 'videos_likes_gt_views': 526,
 'kafka_invalid_timestamp': 1,
 'kafka_invalid_watch_time': 3840,
 'kafka_invalid_interaction': 1,
 'kafka_invalid_device': 1}

In [6]:
grain_audit = {
    "users": {
        "rows": df_users_silver.count(),
        "distinct_ids": df_users_silver.select("user_id").distinct().count(),
    },
    "subscriptions": {
        "rows": df_subscriptions_silver.count(),
        "distinct_ids": df_subscriptions_silver.select("subscription_id").distinct().count(),
    },
    "payments": {
        "rows": df_payments_silver.count(),
        "distinct_ids": df_payments_silver.select("payment_id").distinct().count(),
    },
    "videos": {
        "rows": df_videos_silver.count(),
        "distinct_ids": df_videos_silver.select("video_id").distinct().count(),
    },
    "kafka_events": {
        "rows": df_kafka_silver.count(),
        "distinct_ids": df_kafka_silver.select("event_id").distinct().count(),
    },
}

grain_audit

{'users': {'rows': 10000, 'distinct_ids': 10000},
 'subscriptions': {'rows': 20060, 'distinct_ids': 20060},
 'payments': {'rows': 11784, 'distinct_ids': 11784},
 'videos': {'rows': 10000, 'distinct_ids': 10000},
 'kafka_events': {'rows': 15512, 'distinct_ids': 15512}}

In [7]:
referential_audit = {
    "subscriptions_without_user": (
        df_subscriptions_silver
        .join(df_users_silver.select("user_id"), "user_id", "left_anti")
        .count()
    ),

    "payments_without_subscription": (
        df_payments_silver
        .join(df_subscriptions_silver.select("subscription_id"), "subscription_id", "left_anti")
        .count()
    ),

    "kafka_events_without_user": (
        df_kafka_silver
        .join(df_users_silver.select("user_id"), "user_id", "left_anti")
        .count()
    ),

    "kafka_events_without_video": (
        df_kafka_silver
        .join(df_videos_silver.select("video_id"), "video_id", "left_anti")
        .count()
    ),
}

referential_audit

{'subscriptions_without_user': 0,
 'payments_without_subscription': 0,
 'kafka_events_without_user': 0,
 'kafka_events_without_video': 0}

In [8]:
full_temporal_audit = (
    df_kafka_silver.alias("k")
    .join(
        df_users_silver.select(
            "user_id",
            F.col("created_at").alias("user_created_at")
        ).alias("u"),
        F.col("k.user_id") == F.col("u.user_id"),
        "left"
    )
    .join(
        df_videos_silver.select(
            "video_id",
            F.col("created_at").alias("video_created_at")
        ).alias("v"),
        F.col("k.video_id") == F.col("v.video_id"),
        "left"
    )
    .withColumn(
        "event_before_user",
        F.col("k.event_timestamp") < F.col("u.user_created_at")
    )
    .withColumn(
        "event_before_video",
        F.col("k.event_timestamp") < F.col("v.video_created_at")
    )
)

full_temporal_audit.agg(
    F.count("*").alias("total_events"),
    F.sum(F.when(F.col("event_before_user"), 1).otherwise(0))
        .alias("events_before_user"),
    F.sum(F.when(F.col("event_before_video"), 1).otherwise(0))
        .alias("events_before_video"),
    F.min(
        F.datediff(
            F.to_date("k.event_timestamp"),
            F.to_date("u.user_created_at")
        )
    ).alias("min_days_user_to_event"),
    F.max(
        F.datediff(
            F.to_date("k.event_timestamp"),
            F.to_date("u.user_created_at")
        )
    ).alias("max_days_user_to_event"),
    F.min(
        F.datediff(
            F.to_date("k.event_timestamp"),
            F.to_date("v.video_created_at")
        )
    ).alias("min_days_video_to_event")
).show()

+------------+------------------+-------------------+----------------------+----------------------+-----------------------+
|total_events|events_before_user|events_before_video|min_days_user_to_event|max_days_user_to_event|min_days_video_to_event|
+------------+------------------+-------------------+----------------------+----------------------+-----------------------+
|       15512|                 0|                  0|                     0|                     5|                      1|
+------------+------------------+-------------------+----------------------+----------------------+-----------------------+



In [9]:
user_video_kafka_audit = (
    df_kafka_silver.alias("k")
    .join(
        df_users_silver.select("user_id", "created_at").alias("u"),
        F.col("k.user_id") == F.col("u.user_id"),
        "left"
    )
    .join(
        df_videos_silver.select("video_id", "created_at").alias("v"),
        F.col("k.video_id") == F.col("v.video_id"),
        "left"
    )
    .withColumn(
        "event_before_user",
        F.col("k.event_timestamp") < F.col("u.created_at")
    )
    .withColumn(
        "event_before_video",
        F.col("k.event_timestamp") < F.col("v.created_at")
    )
)

user_video_kafka_audit.agg(
    F.count("*").alias("total_events"),
    F.sum(F.when(F.col("u.user_id").isNull(), 1).otherwise(0)).alias("orphan_users"),
    F.sum(F.when(F.col("v.video_id").isNull(), 1).otherwise(0)).alias("orphan_videos"),
    F.sum(F.when(F.col("event_before_user"), 1).otherwise(0)).alias("events_before_user"),
    F.sum(F.when(F.col("event_before_video"), 1).otherwise(0)).alias("events_before_video")
).show()

+------------+------------+-------------+------------------+-------------------+
|total_events|orphan_users|orphan_videos|events_before_user|events_before_video|
+------------+------------+-------------+------------------+-------------------+
|       15512|           0|            0|                 0|                  0|
+------------+------------+-------------+------------------+-------------------+



In [10]:
video_kafka_audit = (
    df_kafka_silver.alias("k")
    .join(
        df_videos_silver.select("video_id", "created_at").alias("v"),
        F.col("k.video_id") == F.col("v.video_id"),
        "left"
    )
    .withColumn(
        "orphan_video",
        F.col("v.video_id").isNull()
    )
    .withColumn(
        "event_before_video_creation",
        F.col("k.event_timestamp") < F.col("v.created_at")
    )
)

video_kafka_audit.agg(
    F.count("*").alias("total_events"),
    F.sum(F.when(F.col("orphan_video"), 1).otherwise(0)).alias("events_without_matching_video"),
    F.sum(F.when(F.col("event_before_video_creation"), 1).otherwise(0)).alias("events_before_video_creation")
).show()

+------------+-----------------------------+----------------------------+
|total_events|events_without_matching_video|events_before_video_creation|
+------------+-----------------------------+----------------------------+
|       15512|                            0|                           0|
+------------+-----------------------------+----------------------------+



In [11]:
user_kafka_audit = (
    df_kafka_silver.alias("k")
    .join(
        df_users_silver.select("user_id", "created_at").alias("u"),
        F.col("k.user_id") == F.col("u.user_id"),
        "left"
    )
    .withColumn(
        "orphan_user",
        F.col("u.user_id").isNull()
    )
    .withColumn(
        "event_before_user_creation",
        F.col("k.event_timestamp") < F.col("u.created_at")
    )
)

user_kafka_audit.agg(
    F.count("*").alias("total_events"),
    F.sum(F.when(F.col("orphan_user"), 1).otherwise(0)).alias("events_without_matching_user"),
    F.sum(F.when(F.col("event_before_user_creation"), 1).otherwise(0)).alias("events_before_user_creation")
).show()

+------------+----------------------------+---------------------------+
|total_events|events_without_matching_user|events_before_user_creation|
+------------+----------------------------+---------------------------+
|       15512|                           0|                          0|
+------------+----------------------------+---------------------------+



In [12]:
kafka_audit = df_kafka_silver.agg(
    F.count("*").alias("total_events"),
    F.countDistinct("event_id").alias("distinct_event_ids"),
    F.sum(F.when(F.col("event_id").isNull(), 1).otherwise(0)).alias("null_event_ids"),
    F.sum(F.when(F.col("user_id").isNull(), 1).otherwise(0)).alias("null_user_ids"),
    F.sum(F.when(F.col("video_id").isNull(), 1).otherwise(0)).alias("null_video_ids"),
    F.sum(F.when(F.col("event_timestamp").isNull(), 1).otherwise(0)).alias("invalid_timestamps"),
    F.min("event_timestamp").alias("min_event_timestamp"),
    F.max("event_timestamp").alias("max_event_timestamp")
)

kafka_audit.show()

df_kafka_silver.groupBy("event_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+------------+------------------+--------------+-------------+--------------+------------------+-------------------+-------------------+
|total_events|distinct_event_ids|null_event_ids|null_user_ids|null_video_ids|invalid_timestamps|min_event_timestamp|max_event_timestamp|
+------------+------------------+--------------+-------------+--------------+------------------+-------------------+-------------------+
|       15512|             15512|             0|            0|             0|                 1|2026-01-01 00:00:00|2031-05-07 15:13:21|
+------------+------------------+--------------+-------------+--------------+------------------+-------------------+-------------------+

+--------+-----+
|event_id|count|
+--------+-----+
+--------+-----+



In [13]:
user_kafka_audit = (
    df_kafka_silver.alias("k")
    .join(
        df_users_silver.select("user_id", "created_at").alias("u"),
        F.col("k.user_id") == F.col("u.user_id"),
        "left"
    )
    .withColumn(
        "orphan_user",
        F.col("u.user_id").isNull()
    )
    .withColumn(
        "event_before_user_creation",
        F.col("k.event_timestamp") < F.col("u.created_at")
    )
)

user_kafka_audit.agg(
    F.count("*").alias("total_events"),
    F.sum(F.when(F.col("orphan_user"), 1).otherwise(0)).alias("events_without_matching_user"),
    F.sum(F.when(F.col("event_before_user_creation"), 1).otherwise(0)).alias("events_before_user_creation")
).show()

+------------+----------------------------+---------------------------+
|total_events|events_without_matching_user|events_before_user_creation|
+------------+----------------------------+---------------------------+
|       15512|                           0|                          0|
+------------+----------------------------+---------------------------+



In [14]:
video_audit = df_videos_silver.agg(
    F.count("*").alias("total_videos"),
    F.countDistinct("video_id").alias("distinct_video_ids"),
    F.sum(F.when(F.col("video_id").isNull(), 1).otherwise(0)).alias("null_video_ids"),
    F.sum(F.when(F.col("match_id").isNull(), 1).otherwise(0)).alias("null_match_ids"),
    F.sum(
        F.when(
            F.col("last_updated") < F.col("created_at"),
            1
        ).otherwise(0)
    ).alias("last_updated_before_created"),
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at")
)

video_audit.show()

+------------+------------------+--------------+--------------+---------------------------+-------------------+-------------------+
|total_videos|distinct_video_ids|null_video_ids|null_match_ids|last_updated_before_created|     min_created_at|     max_created_at|
+------------+------------------+--------------+--------------+---------------------------+-------------------+-------------------+
|       10000|             10000|             0|             1|                          0|2025-12-01 00:01:00|2025-12-31 23:59:00|
+------------+------------------+--------------+--------------+---------------------------+-------------------+-------------------+



In [15]:
df_videos_silver.groupBy("video_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+--------+-----+
|video_id|count|
+--------+-----+
+--------+-----+



In [16]:
chain_audit = (
    df_payments_silver.alias("p")
    .join(
        df_subscriptions_silver.select(
            "subscription_id",
            "user_id",
            "start_date",
            "end_date"
        ).alias("s"),
        F.col("p.subscription_id") == F.col("s.subscription_id"),
        "left"
    )
    .join(
        df_users_silver.select(
            "user_id",
            "created_at"
        ).alias("u"),
        F.col("s.user_id") == F.col("u.user_id"),
        "left"
    )
)

chain_audit.agg(
    F.count("*").alias("total_payments"),
    F.sum(F.when(F.col("u.user_id").isNull(), 1).otherwise(0)).alias("broken_user_chain"),
    F.sum(
        F.when(
            F.col("p.payment_date") < F.col("s.start_date"),
            1
        ).otherwise(0)
    ).alias("payment_before_subscription"),
    F.sum(
        F.when(
            F.col("p.payment_date") < F.col("u.created_at"),
            1
        ).otherwise(0)
    ).alias("payment_before_user_creation")
).show()

+--------------+-----------------+---------------------------+----------------------------+
|total_payments|broken_user_chain|payment_before_subscription|payment_before_user_creation|
+--------------+-----------------+---------------------------+----------------------------+
|         11784|                0|                          0|                           0|
+--------------+-----------------+---------------------------+----------------------------+



In [17]:
payment_audit = (
    df_payments_silver.alias("p")
    .join(
        df_subscriptions_silver.select(
            "subscription_id",
            "start_date",
            "end_date"
        ).alias("s"),
        F.col("p.subscription_id") == F.col("s.subscription_id"),
        "left"
    )
    .withColumn(
        "orphan_subscription",
        F.col("s.subscription_id").isNull()
    )
    .withColumn(
        "payment_before_subscription",
        F.col("p.payment_date") < F.col("s.start_date")
    )
    .withColumn(
        "payment_after_subscription",
        F.col("s.end_date").isNotNull()
        & (F.col("p.payment_date") > F.col("s.end_date"))
    )
)

payment_audit.agg(
    F.count("*").alias("total_payments"),
    F.countDistinct("p.payment_id").alias("distinct_payment_ids"),
    F.sum(F.when(F.col("p.payment_id").isNull(), 1).otherwise(0)).alias("null_payment_ids"),
    F.sum(F.when(F.col("orphan_subscription"), 1).otherwise(0)).alias("orphan_subscriptions"),
    F.sum(F.when(F.col("payment_before_subscription"), 1).otherwise(0)).alias("payment_before_subscription"),
    F.sum(F.when(F.col("payment_after_subscription"), 1).otherwise(0)).alias("payment_after_subscription")
).show()

+--------------+--------------------+----------------+--------------------+---------------------------+--------------------------+
|total_payments|distinct_payment_ids|null_payment_ids|orphan_subscriptions|payment_before_subscription|payment_after_subscription|
+--------------+--------------------+----------------+--------------------+---------------------------+--------------------------+
|         11784|               11784|               0|                   0|                          0|                         0|
+--------------+--------------------+----------------+--------------------+---------------------------+--------------------------+



In [18]:
df_payments_silver.groupBy("payment_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+----------+-----+
|payment_id|count|
+----------+-----+
+----------+-----+



In [19]:
subscription_audit = (
    df_subscriptions_silver.alias("s")
    .join(
        df_users_silver.select("user_id", "created_at").alias("u"),
        F.col("s.user_id") == F.col("u.user_id"),
        "left"
    )
    .withColumn(
        "orphan_user",
        F.col("u.user_id").isNull()
    )
    .withColumn(
        "subscription_before_user",
        F.col("s.start_date") < F.col("u.created_at")
    )
    .withColumn(
        "end_before_start",
        F.col("s.end_date").isNotNull()
        & (F.col("s.end_date") < F.col("s.start_date"))
    )
)

subscription_audit.agg(
    F.count("*").alias("total_subscriptions"),
    F.countDistinct("s.subscription_id").alias("distinct_subscription_ids"),
    F.sum(
        F.when(F.col("s.subscription_id").isNull(), 1).otherwise(0)
    ).alias("null_subscription_ids"),
    F.sum(
        F.when(F.col("orphan_user"), 1).otherwise(0)
    ).alias("orphan_users"),
    F.sum(
        F.when(F.col("subscription_before_user"), 1).otherwise(0)
    ).alias("subscription_before_user_creation"),
    F.sum(
        F.when(F.col("end_before_start"), 1).otherwise(0)
    ).alias("end_before_start")
).show()

+-------------------+-------------------------+---------------------+------------+---------------------------------+----------------+
|total_subscriptions|distinct_subscription_ids|null_subscription_ids|orphan_users|subscription_before_user_creation|end_before_start|
+-------------------+-------------------------+---------------------+------------+---------------------------------+----------------+
|              20060|                    20060|                    0|           0|                                0|               1|
+-------------------+-------------------------+---------------------+------------+---------------------------------+----------------+



In [20]:
df_subscriptions_silver.groupBy("subscription_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+---------------+-----+
|subscription_id|count|
+---------------+-----+
+---------------+-----+



In [30]:
user_audit = df_users_silver.agg(
    F.count("*").alias("total_users"),
    F.countDistinct("user_id").alias("distinct_user_ids"),
    F.sum(
        F.when(F.col("user_id").isNull(), 1).otherwise(0)
    ).alias("null_user_ids"),
    F.sum(
        F.when(F.col("created_at").isNull(), 1).otherwise(0)
    ).alias("null_created_at"),
    F.sum(
        F.when(
            F.col("last_login").isNotNull()
            & (F.col("last_login") < F.col("created_at")),
            1
        ).otherwise(0)
    ).alias("login_before_creation"),
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at"),
    F.countDistinct("created_at").alias("distinct_created_at")
)

user_audit.show()

+-----------+-----------------+-------------+---------------+---------------------+-------------------+-------------------+-------------------+
|total_users|distinct_user_ids|null_user_ids|null_created_at|login_before_creation|     min_created_at|     max_created_at|distinct_created_at|
+-----------+-----------------+-------------+---------------+---------------------+-------------------+-------------------+-------------------+
|      10000|            10000|            0|              0|                    1|2026-01-01 00:00:00|2031-05-24 00:23:13|              10000|
+-----------+-----------------+-------------+---------------+---------------------+-------------------+-------------------+-------------------+



In [4]:
df_users_silver.groupBy("user_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+-------+-----+
|user_id|count|
+-------+-----+
+-------+-----+



In [5]:
df_users_silver.select(
    "user_id",
    "created_at",
    "last_login",
    "account_status"
).filter(
    F.col("last_login").isNotNull()
).orderBy("created_at").show(20, truncate=False)

+------------------------------------+-------------------+-------------------+--------------+
|user_id                             |created_at         |last_login         |account_status|
+------------------------------------+-------------------+-------------------+--------------+
|01a06562-ea0a-7292-be01-8f7b75834952|2026-01-01 01:25:00|2025-12-31 01:25:00|active        |
+------------------------------------+-------------------+-------------------+--------------+



In [13]:
POSTGRES_RUN_PATH = (
    "s3a://end-to-end-streaming-data-platform-bronze/"
    "postgres/ingestion_date=2026-09-03-Sep"
)

In [14]:
df_users_silver = spark.read.parquet(
    f"{POSTGRES_RUN_PATH}/users.parquet"
)

df_subscriptions_silver = spark.read.parquet(
    f"{POSTGRES_RUN_PATH}/subscription.parquet"
)

df_payments_silver = spark.read.parquet(
    f"{POSTGRES_RUN_PATH}/payments.parquet"
)

In [27]:


# Mongo Silver
videos = df_videos_silver.select(
    "video_id",
    "created_at"
)

# Kafka Silver
events = df_kafka_silver.select(
    "event_id",
    "video_id",
    "event_timestamp",
    "interaction_type"
)


temporal_check = (
    events.alias("e")
    .join(
        videos.alias("v"),
        F.col("e.video_id") == F.col("v.video_id"),
        "left"
    )
    .withColumn(
        "event_before_video_created",
        F.col("e.event_timestamp") < F.col("v.created_at")
    )
)

# 
temporal_check.agg(
    F.count("*").alias("total_events"),
    F.sum(
        F.when(F.col("v.video_id").isNull(), 1).otherwise(0)
    ).alias("events_without_matching_video"),
    F.sum(
        F.when(F.col("event_before_video_created"), 1).otherwise(0)
    ).alias("events_before_video_created")
).show()
print('=' *80)


+------------+-----------------------------+---------------------------+
|total_events|events_without_matching_video|events_before_video_created|
+------------+-----------------------------+---------------------------+
|       15512|                            0|                          0|
+------------+-----------------------------+---------------------------+



In [28]:
temporal_check.filter(
    F.col("event_before_video_created") |
    F.col("v.video_id").isNull()
).select(
    "event_id",
    F.col("e.video_id").alias("video_id"),
    "created_at",
    "event_timestamp",
    "interaction_type"
).orderBy(
    F.col("event_timestamp")
).show(50, truncate=False)

+--------+--------+----------+---------------+----------------+
|event_id|video_id|created_at|event_timestamp|interaction_type|
+--------+--------+----------+---------------+----------------+
+--------+--------+----------+---------------+----------------+



In [29]:
video_event_gap = (
    temporal_check
    .filter(F.col("v.video_id").isNotNull())
    .withColumn(
        "minutes_after_video_creation",
        (
            F.col("e.event_timestamp").cast("long")
            - F.col("v.created_at").cast("long")
        ) / 60
    )
)

video_event_gap.select(
    F.min("minutes_after_video_creation").alias("min_gap_minutes"),
    F.max("minutes_after_video_creation").alias("max_gap_minutes"),
    F.avg("minutes_after_video_creation").alias("avg_gap_minutes")
).show()

+------------------+---------------+-----------------+
|   min_gap_minutes|max_gap_minutes|  avg_gap_minutes|
+------------------+---------------+-----------------+
|1099.1333333333334|      2853823.5|762929.1544592014|
+------------------+---------------+-----------------+



In [25]:


# =========================
# USERS
# =========================

print("=== USERS ===")

df_users_silver.select(
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at"),
    F.min("last_login").alias("min_last_login"),
    F.max("last_login").alias("max_last_login"),
).show(truncate=False)

# Login cannot be before user creation
users_invalid_login = df_users_silver.filter(
    F.col("last_login").isNotNull()
    & (F.col("last_login") < F.col("created_at"))
)

print(
    "Users with last_login before created_at:",
    users_invalid_login.count()
)


# =========================
# SUBSCRIPTIONS
# =========================

print("=== SUBSCRIPTIONS ===")

df_subscriptions_silver.select(
    F.min("start_date").alias("min_start_date"),
    F.max("start_date").alias("max_start_date"),
    F.min("end_date").alias("min_end_date"),
    F.max("end_date").alias("max_end_date"),
).show(truncate=False)

# End cannot be before start
subscriptions_invalid_dates = df_subscriptions_silver.filter(
    F.col("end_date").isNotNull()
    & (F.col("end_date") < F.col("start_date"))
)

print(
    "Subscriptions with end_date before start_date:",
    subscriptions_invalid_dates.count()
)


# =========================
# PAYMENTS
# =========================

print("=== PAYMENTS ===")

df_payments_silver.select(
    F.min("payment_date").alias("min_payment_date"),
    F.max("payment_date").alias("max_payment_date"),
).show(truncate=False)


# =========================
# CROSS-SOURCE TEMPORAL CHECKS
# =========================

print("=== CROSS-SOURCE ===")

# Payment must belong temporally to its subscription
df_payment_subscription = (
    df_payments_silver.alias("p")
    .join(
        df_subscriptions_silver.alias("s"),
        F.col("p.subscription_id")
        == F.col("s.subscription_id"),
        "left",
    )
)

payments_before_subscription = df_payment_subscription.filter(
    F.col("p.payment_date") < F.col("s.start_date")
)

print(
    "Payments before subscription start:",
    payments_before_subscription.count()
)

payments_after_subscription = df_payment_subscription.filter(
    F.col("s.end_date").isNotNull()
    & (F.col("p.payment_date") > F.col("s.end_date"))
)

print(
    "Payments after subscription end:",
    payments_after_subscription.count()
)


# =========================
# USER → SUBSCRIPTION
# =========================

df_user_subscription = (
    df_subscriptions_silver.alias("s")
    .join(
        df_users_silver.alias("u"),
        F.col("s.user_id") == F.col("u.user_id"),
        "left",
    )
)

subscriptions_before_user = df_user_subscription.filter(
    F.col("s.start_date") < F.col("u.created_at")
)

print(
    "Subscriptions before user creation:",
    subscriptions_before_user.count()
)

=== USERS ===
+-------------------+-------------------+-------------------+-------------------+
|min_created_at     |max_created_at     |min_last_login     |max_last_login     |
+-------------------+-------------------+-------------------+-------------------+
|2026-01-01 00:00:00|2031-05-24 00:23:13|2025-12-31 02:04:12|2025-12-31 02:04:12|
+-------------------+-------------------+-------------------+-------------------+

Users with last_login before created_at: 1
=== SUBSCRIPTIONS ===
+-------------------+-------------------+-------------------+-------------------+
|min_start_date     |max_start_date     |min_end_date       |max_end_date       |
+-------------------+-------------------+-------------------+-------------------+
|2026-01-01 00:00:00|2031-05-24 00:23:13|2025-12-31 00:00:00|2032-05-22 22:42:12|
+-------------------+-------------------+-------------------+-------------------+

Subscriptions with end_date before start_date: 1
=== PAYMENTS ===
+-------------------+------------

In [16]:
# ============================================================
# 3. BASE COUNTS
# ============================================================

base_counts = {
    "users": df_users.count(),
    "subscriptions": df_subscriptions.count(),
    "payments": df_payments.count(),
    "videos": df_videos.count(),
    "interactions": df_interactions.count(),
}

base_counts

NameError: name 'df_users' is not defined

In [5]:
# ============================================================
# 4. SUBSCRIPTION -> USER
# ============================================================

subscription_user_check = (
    df_subscriptions.alias("s")
    .join(
        df_users.alias("u"),
        F.col("s.user_id") == F.col("u.user_id"),
        "left"
    )
    .select(
        F.col("s.subscription_id"),
        F.col("s.user_id"),
        F.col("s.start_date"),
        F.col("u.created_at").alias("user_created_at")
    )
    .withColumn(
        "valid_user_relationship",
        F.col("user_created_at").isNotNull()
    )
    .withColumn(
        "valid_subscription_timing",
        F.col("user_created_at").isNotNull()
        & (F.col("s.start_date") >= F.col("user_created_at"))
    )
)

In [6]:
subscription_user_summary = subscription_user_check.agg(
    F.count("*").alias("total_subscriptions"),
    F.sum(
        F.when(F.col("valid_user_relationship"), 1).otherwise(0)
    ).alias("valid_user_relationships"),
    F.sum(
        F.when(F.col("valid_subscription_timing"), 1).otherwise(0)
    ).alias("valid_subscription_timing")
)

subscription_user_summary.show()

+-------------------+------------------------+-------------------------+
|total_subscriptions|valid_user_relationships|valid_subscription_timing|
+-------------------+------------------------+-------------------------+
|              20032|                   20032|                    20032|
+-------------------+------------------------+-------------------------+



In [7]:
subscription_user_check.filter(
    ~F.col("valid_subscription_timing")
).show(10, truncate=False, vertical=True)

(0 rows)



In [8]:
# ============================================================
# 5. PAYMENT -> SUBSCRIPTION
# ============================================================

payment_subscription_check = (
    df_payments.alias("p")
    .join(
        df_subscriptions.alias("s"),
        F.col("p.subscription_id") == F.col("s.subscription_id"),
        "left"
    )
    .select(
        F.col("p.payment_id"),
        F.col("p.subscription_id"),
        F.col("p.payment_date"),
        F.col("s.start_date"),
        F.col("s.end_date")
    )
    .withColumn(
        "valid_subscription_relationship",
        F.col("start_date").isNotNull()
    )
    .withColumn(
        "valid_payment_after_start",
        F.col("start_date").isNotNull()
        & (F.col("payment_date") >= F.col("start_date"))
    )
    .withColumn(
        "valid_payment_before_end",
        F.col("end_date").isNull()
        | (F.col("payment_date") <= F.col("end_date"))
    )
)

In [9]:
payment_subscription_check.agg(
    F.count("*").alias("total_payments"),
    F.sum(
        F.when(F.col("valid_subscription_relationship"), 1).otherwise(0)
    ).alias("valid_subscription_relationships"),
    F.sum(
        F.when(F.col("valid_payment_after_start"), 1).otherwise(0)
    ).alias("payments_after_start"),
    F.sum(
        F.when(F.col("valid_payment_before_end"), 1).otherwise(0)
    ).alias("payments_before_end")
).show()

+--------------+--------------------------------+--------------------+-------------------+
|total_payments|valid_subscription_relationships|payments_after_start|payments_before_end|
+--------------+--------------------------------+--------------------+-------------------+
|         11771|                           11771|               11771|              11771|
+--------------+--------------------------------+--------------------+-------------------+



In [10]:
# ============================================================
# 6. INTERACTION -> USER
# ============================================================

interaction_user_check = (
    df_interactions.alias("i")
    .join(
        df_users.alias("u"),
        F.col("i.user_id") == F.col("u.user_id"),
        "left"
    )
    .select(
        F.col("i.event_id"),
        F.col("i.user_id"),
        F.col("i.event_timestamp"),
        F.col("u.created_at").alias("user_created_at")
    )
    .withColumn(
        "valid_user_relationship",
        F.col("user_created_at").isNotNull()
    )
    .withColumn(
        "valid_interaction_after_user_creation",
        F.col("user_created_at").isNotNull()
        & (F.col("event_timestamp") >= F.col("user_created_at"))
    )
)

In [11]:
interaction_user_check.agg(
    F.count("*").alias("total_interactions"),
    F.sum(
        F.when(F.col("valid_user_relationship"), 1).otherwise(0)
    ).alias("valid_user_relationships"),
    F.sum(
        F.when(F.col("valid_interaction_after_user_creation"), 1)
        .otherwise(0)
    ).alias("valid_interaction_timing")
).show()

+------------------+------------------------+------------------------+
|total_interactions|valid_user_relationships|valid_interaction_timing|
+------------------+------------------------+------------------------+
|             16074|                   16074|                   16073|
+------------------+------------------------+------------------------+



In [12]:
# ============================================================
# 7. INTERACTION -> VIDEO
# ============================================================

interaction_video_check = (
    df_interactions.alias("i")
    .join(
        df_videos.alias("v"),
        F.col("i.video_id") == F.col("v.video_id"),
        "left"
    )
    .select(
        F.col("i.event_id"),
        F.col("i.video_id"),
        F.col("i.event_timestamp"),
        F.col("v.created_at").alias("video_created_at")
    )
    .withColumn(
        "valid_video_relationship",
        F.col("video_created_at").isNotNull()
    )
    .withColumn(
        "valid_interaction_after_video_creation",
        F.col("video_created_at").isNotNull()
        & (F.col("event_timestamp") >= F.col("video_created_at"))
    )
)

In [13]:
interaction_video_check.agg(
    F.count("*").alias("total_interactions"),
    F.sum(
        F.when(F.col("valid_video_relationship"), 1).otherwise(0)
    ).alias("valid_video_relationships"),
    F.sum(
        F.when(F.col("valid_interaction_after_video_creation"), 1)
        .otherwise(0)
    ).alias("valid_interaction_timing")
).show()

+------------------+-------------------------+------------------------+
|total_interactions|valid_video_relationships|valid_interaction_timing|
+------------------+-------------------------+------------------------+
|             16074|                    16074|                   16073|
+------------------+-------------------------+------------------------+



In [14]:
interaction_video_check.filter(
    ~F.col("valid_interaction_after_video_creation")
).show(5, truncate=False, vertical=True)

(0 rows)



In [15]:
# ============================================================
# 8. FULL USER -> SUBSCRIPTION -> PAYMENT CHAIN
# ============================================================

user_subscription_payment = (
    df_payments.alias("p")

    .join(
        df_subscriptions.alias("s"),
        F.col("p.subscription_id") == F.col("s.subscription_id"),
        "inner"
    )

    .join(
        df_users.alias("u"),
        F.col("s.user_id") == F.col("u.user_id"),
        "inner"
    )

    .select(
        F.col("u.user_id"),
        F.col("u.created_at").alias("user_created_at"),

        F.col("s.subscription_id"),
        F.col("s.start_date").alias("subscription_start"),
        F.col("s.end_date").alias("subscription_end"),

        F.col("p.payment_id"),
        F.col("p.payment_date")
    )

    .withColumn(
        "valid_user_to_subscription",
        F.col("subscription_start") >= F.col("user_created_at")
    )

    .withColumn(
        "valid_payment_after_subscription_start",
        F.col("payment_date") >= F.col("subscription_start")
    )

    .withColumn(
        "valid_payment_before_subscription_end",
        F.col("subscription_end").isNull()
        | (F.col("payment_date") <= F.col("subscription_end"))
    )
)

In [16]:
user_subscription_payment.agg(
    F.count("*").alias("payment_records"),

    F.sum(
        F.when(
            F.col("valid_user_to_subscription"), 1
        ).otherwise(0)
    ).alias("valid_user_subscription"),

    F.sum(
        F.when(
            F.col("valid_payment_after_subscription_start"), 1
        ).otherwise(0)
    ).alias("valid_payment_start_timing"),

    F.sum(
        F.when(
            F.col("valid_payment_before_subscription_end"), 1
        ).otherwise(0)
    ).alias("valid_payment_end_timing")
).show()

+---------------+-----------------------+--------------------------+------------------------+
|payment_records|valid_user_subscription|valid_payment_start_timing|valid_payment_end_timing|
+---------------+-----------------------+--------------------------+------------------------+
|          11771|                  11771|                     11771|                   11771|
+---------------+-----------------------+--------------------------+------------------------+



In [17]:
# ============================================================
# 9. VIDEO TEMPORAL CONSISTENCY
# ============================================================

video_temporal_check = (
    df_videos
    .withColumn(
        "valid_video_timestamps",
        F.col("last_updated") >= F.col("created_at")
    )
)

video_temporal_check.agg(
    F.count("*").alias("total_videos"),
    F.sum(
        F.when(F.col("valid_video_timestamps"), 1).otherwise(0)
    ).alias("valid_video_timestamps")
).show()

+------------+----------------------+
|total_videos|valid_video_timestamps|
+------------+----------------------+
|       10000|                 10000|
+------------+----------------------+



In [24]:
# ============================================================
# 10. GRAIN CHECK
# ============================================================

interaction_joined = (
    df_interactions.alias("i")
    .join(
        df_users.alias("u"),
        F.col("i.user_id") == F.col("u.user_id"),
        "left"
    )
    .join(
        df_videos.alias("v"),
        F.col("i.video_id") == F.col("v.video_id"),
        "left"
    )
)

print("Original interactions:", df_interactions.count())
print("Joined interactions:", interaction_joined.count())

NameError: name 'df_interactions' is not defined

In [22]:
df_users_silver.select(
    F.min("created_at").alias("min_created_at"),
    F.max("created_at").alias("max_created_at"),
    F.countDistinct("created_at").alias("distinct_created_at")
).show()

+-------------------+-------------------+-------------------+
|     min_created_at|     max_created_at|distinct_created_at|
+-------------------+-------------------+-------------------+
|2026-01-01 00:00:00|2031-05-24 00:23:13|              10000|
+-------------------+-------------------+-------------------+



In [23]:
df_users_silver.select(
    "user_id",
    "created_at"
).orderBy("created_at").show(20, truncate=False)

+------------------------------------+-------------------+
|user_id                             |created_at         |
+------------------------------------+-------------------+
|01a0687c-0cc4-7119-a4fa-ca2afba17a91|2026-01-01 00:00:00|
|01a0687c-0ccc-7875-bf34-6d694b2b97fc|2026-01-01 01:02:31|
|01a0687c-0cd3-7a5c-a16c-3635d31b372b|2026-01-01 02:04:12|
|01a0687c-0cd9-711b-b731-edba851af43a|2026-01-01 03:07:40|
|01a0687c-0ce1-7610-bbe7-165b98e1291a|2026-01-01 03:59:08|
|01a0687c-0ce8-74cf-a3e5-19885d791ff9|2026-01-01 05:00:03|
|01a0687c-0cf2-7b8f-82f1-be1ef67b439b|2026-01-01 17:49:21|
|01a0687c-0cfb-7a06-baab-2699344cfd53|2026-01-01 18:45:21|
|01a0687c-0d03-7099-acc3-2dd6fdf9a1b8|2026-01-01 19:35:01|
|01a0687c-0d09-7e3f-9c1f-a306c98905a7|2026-01-01 20:17:40|
|01a0687c-0d0f-7f9d-b4e9-62b22a48cf47|2026-01-01 21:00:44|
|01a0687c-0d16-7c9c-b14d-b220681affe6|2026-01-01 21:32:59|
|01a0687c-0d1d-752e-8e60-8194a51059fc|2026-01-01 22:22:30|
|01a0687c-0d25-7314-9a05-58440c71c5bc|2026-01-03 23:19:0

In [21]:
user_subscription_check = (
    df_subscriptions_silver.alias("s")
    .join(
        df_users_silver.alias("u"),
        F.col("s.user_id") == F.col("u.user_id"),
        "left"
    )
    .withColumn(
        "subscription_before_user",
        F.col("s.start_date") < F.col("u.created_at")
    )
)

user_subscription_check.agg(
    F.count("*").alias("total_subscriptions"),
    F.sum(
        F.when(F.col("subscription_before_user"), 1).otherwise(0)
    ).alias("subscriptions_before_user_creation")
).show()

+-------------------+----------------------------------+
|total_subscriptions|subscriptions_before_user_creation|
+-------------------+----------------------------------+
|              20060|                                 0|
+-------------------+----------------------------------+

